# Build a question that can distinguish two translations

A message says to notify someone after an application is approved. Changing *approved* to *rejected* changes when you should send the notification. If the application has just been approved, one short question has opposite answers for those two messages.

## What the reviewer checks

This notebook shows how we designed and checked these questions for our Slavic-to-English translation project. For each synthetic situation we wrote two source messages, changed one word, drafted an English reference for each and wrote one English yes/no question that both versions can answer. A language reviewer checked the wording and the intended decisions. The gold answers are the recorded judgments. A one-word edit alone cannot establish that the decisions are correct.

### The 36 scenarios in B01

Batch **B01** contains **36 training scenarios**: 12 each in Polish, Croatian and Russian. Each has two variants, for 72 source–English pairs. The mixed arm of the 384-pair Gemma pilot used them for training. The 300,000-pair Gemma run used official text only. Ukrainian was held out of fine-tuning. The *historical Everyday72* and *Stress16* are separate regression sets. Once B01 enters training, measuring performance on B01 cannot establish generalization to new situations.

## Run three reviewed examples on CPU

The examples below work standalone in Colab. The complete batch is in `data/b01_contrasts.jsonl` in the repository, with one JSON object per scenario. Variants contain approved source text and English references. Questions and their keys come from the design records used for review. RU-05 includes the author's final revision, which specifies that the notification is sent only *after* the decision.

Start with the **proposed action**. For the honey scenario, a stallholder has seven jars to put in each box. If the rule says *at most six*, the answer is no. If it says *at least six*, the answer is yes. Choosing six jars would give the same answer for both versions and make the question useless for detecting that mistake.

In [1]:
import json
from collections import Counter
from pathlib import Path
import random
import re
import unicodedata


EXAMPLES = [
    {
        "scenario_id": "b01-ru-05",
        "lang": "ru",
        "category": "prerequisite",
        "question_en": "The application has been approved. Does that satisfy the condition for messaging the speaker?",
        "variants": {
            "A": {
                "source": "Напиши мне про заявку на пропуск только после того, как её одобрят.",
                "reference_en": "Message me about the pass application only after it’s been approved.",
                "expected_answer": "yes",
            },
            "B": {
                "source": "Напиши мне про заявку на пропуск только после того, как её отклонят.",
                "reference_en": "Message me about the pass application only after it’s been rejected.",
                "expected_answer": "no",
            },
        },
    },
    {
        "scenario_id": "b01-pl-02",
        "lang": "pl",
        "category": "time",
        "question_en": "Can I use the basement laundry room on the seventh of the month?",
        "variants": {
            "A": {
                "source": "Z pralni w piwnicy mogę korzystać tylko w parzyste dni miesiąca.",
                "reference_en": "I can only use the basement laundry room on even-numbered days of the month.",
                "expected_answer": "no",
            },
            "B": {
                "source": "Z pralni w piwnicy mogę korzystać tylko w nieparzyste dni miesiąca.",
                "reference_en": "I can only use the basement laundry room on odd-numbered days of the month.",
                "expected_answer": "yes",
            },
        },
    },
    {
        "scenario_id": "b01-hr-03",
        "lang": "hr",
        "category": "quantity",
        "question_en": "Does packing seven jars in every box meet the instruction?",
        "variants": {
            "A": {
                "source": "U svaku kutiju za štand stavi najviše šest staklenki meda.",
                "reference_en": "Put at most six jars of honey in each box for the stall.",
                "expected_answer": "no",
            },
            "B": {
                "source": "U svaku kutiju za štand stavi najmanje šest staklenki meda.",
                "reference_en": "Put at least six jars of honey in each box for the stall.",
                "expected_answer": "yes",
            },
        },
    },
]

## Load and validate your questions

Leave `INPUT_JSONL` as `None` for the three embedded examples. From a repository checkout, point it to the included `data/b01_contrasts.jsonl` relative to your kernel's working directory, or supply your own file with the same fields. An optional `family_id` can link paraphrases or cross-language versions of the same situation.

The check uses Unicode NFC, case folding and whitespace normalization for comparison only. Punctuation and numbers are tokens and must stay the same; the single permitted replacement must be a word. Source and reference strings in the output retain their original spelling. The checker also requires two distinct English references and opposite recorded answers to one question. A bilingual reader still needs to judge the actual change of meaning.

In [2]:
INPUT_JSONL = None


def load_scenarios(path):
    if path is None:
        return EXAMPLES
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def normalized(text):
    return " ".join(unicodedata.normalize("NFC", text).casefold().split())


def tokens(text):
    return re.findall(r"[^\W\d_]+|\d+|[^\w\s]|_", normalized(text))


def changed_words(first, second):
    a, b = tokens(first), tokens(second)
    if len(a) != len(b):
        raise ValueError("The sources have different token counts.")
    edits = [(x, y) for x, y in zip(a, b) if x != y]
    if len(edits) != 1 or not all(word.isalpha() for word in edits[0]):
        raise ValueError("Expected one word substitution with punctuation and numbers fixed.")
    return edits[0]


def validate(scenarios):
    seen = set()
    changes = {}
    if not scenarios:
        raise ValueError("Provide at least one complete scenario.")
    for scenario in scenarios:
        scenario_id = scenario["scenario_id"]
        variants = scenario["variants"]
        if not scenario_id.strip() or scenario_id in seen or set(variants) != {"A", "B"}:
            raise ValueError(f"{scenario_id}: expected a unique ID and variants A and B.")
        seen.add(scenario_id)
        if not all(scenario[key].strip() for key in ("lang", "category", "question_en")):
            raise ValueError(f"{scenario_id}: missing language, category or question.")
        if {variant["expected_answer"] for variant in variants.values()} != {"yes", "no"}:
            raise ValueError(f"{scenario_id}: answers must be opposite.")
        refs = {normalized(variant["reference_en"]) for variant in variants.values()}
        if len(refs) != 2 or "" in refs:
            raise ValueError(f"{scenario_id}: provide two distinct English references.")
        changes[scenario_id] = changed_words(variants["A"]["source"], variants["B"]["source"])
    return changes


scenarios = load_scenarios(INPUT_JSONL)
changes = validate(scenarios)
print(f"{len(scenarios)} scenarios, {2 * len(scenarios)} translation pairs")
print("Scenarios by language:", dict(sorted(Counter(s["lang"] for s in scenarios).items())))
for scenario in scenarios[:3]:
    before, after = changes[scenario["scenario_id"]]
    print(f"{scenario['scenario_id']}: {before} → {after}")

3 scenarios, 6 translation pairs
Scenarios by language: {'hr': 1, 'pl': 1, 'ru': 1}
b01-ru-05: одобрят → отклонят
b01-pl-02: parzyste → nieparzyste
b01-hr-03: najviše → najmanje


## Keep the scenario intact

For our training export every B01 variant belonged to the training partition. We carried the scenario ID with both rows. If new versions describe the same situation, assign them the same `family_id` *before* selecting train, development or test partitions. Split at the family level so that a wording variant cannot leak into another partition.

This produces ordinary source–target rows with their group ID. The training prompt contains the source message, and the English reference supplies the target. Keep the decision question and its answer key in the review material.

In [3]:
training_rows = []
for scenario in scenarios:
    group = scenario.get("family_id", scenario["scenario_id"])
    for variant_name, variant in scenario["variants"].items():
        training_rows.append({
            "scenario_id": scenario["scenario_id"],
            "group": group,
            "variant": variant_name,
            "lang": scenario["lang"],
            "source": variant["source"],
            "target": variant["reference_en"],
        })

print(f"{len(training_rows)} source–target rows across "
      f"{len({row['group'] for row in training_rows})} scenario groups")

6 source–target rows across 3 scenario groups


## Present one version to the English-speaking recipient

A reviewer of *source quality* may inspect both variants. A recipient performing the *comprehension test* gets only one English message and the fixed question, with no source text, answer, variant or system label. Pass only the displayed card to the recipient. This demonstration uses the approved English reference to check the question. In a model comparison pass the model's translated text in its place and keep the blinded assignment with the organizer. Assign only one version of a situation to each recipient.

In [4]:
def recipient_card(english_message, question):
    return {"message": english_message, "question": question}


scenario = scenarios[0]
variant_name = random.Random(42).choice(sorted(scenario["variants"]))
variant = scenario["variants"][variant_name]
card = recipient_card(variant["reference_en"], scenario["question_en"])
print(card["message"])
print(card["question"])

Message me about the pass application only after it’s been approved.
The application has been approved. Does that satisfy the condition for messaging the speaker?


## Check the answer separately

Only the organizer opens the answer key after the response is recorded. If a translation makes the decision unclear, keep the response as *uncertain*. Counting it as yes or no would hide the ambiguity.

In [5]:
organizer_key = {
    "scenario": scenario["scenario_id"],
    "variant": variant_name,
    "expected_answer": variant["expected_answer"],
}
print("Organizer's key:", organizer_key)
print("Recorded decisions for this question:",
      {label: item["expected_answer"] for label, item in scenario["variants"].items()})

Organizer's key: {'scenario': 'b01-ru-05', 'variant': 'A', 'expected_answer': 'yes'}
Recorded decisions for this question: {'A': 'yes', 'B': 'no'}


The three built-in scenarios illustrate the rule. Across the full batch we used six types of change: time, quantity, prerequisite, exception, permission and scope. To assess usefulness on new messages, reserve fresh situations before choosing a model or changing its prompt. Keep the public B01 examples available for training and demonstrations.